In [3]:
import requests
import os
from dotenv import load_dotenv

#wczytanie klucza api
load_dotenv()
API_KEY = os.getenv('API_KEY')

URL = f"https://api.nasa.gov/neo/rest/v1/feed?start_date=2026-05-04&end_date=2026-05-04&api_key={API_KEY}"

#zapytanie do serwera

response = requests.get(URL)

print("status połączenia:", response.status_code)

status połączenia: 200


In [5]:
dane = response.json()

print("Główne sekcje w paczce to:", dane.keys())


Główne sekcje w paczce to: dict_keys(['links', 'element_count', 'near_earth_objects'])


In [11]:
#sprawdzamy łączną liczbę obiektów
liczba_asteroid = dane['element_count']
print(f"Liczba asteroid zidentyfikowanych dzisiaj: {liczba_asteroid}")

#liczba asteroid dla konkretnej daty
data_dzisiajsza = "2026-05-04" #yyy-mm-dd
lista_asteroid = dane['near_earth_objects'][data_dzisiajsza]

#pierwszy element na liście
pierwsza_asteroida = lista_asteroid[0]

#sprawdzenie udostepnionych parametrow pierwszej asteroidy
print("Dane ukryte w pierwszej asteroidze:")
print(pierwsza_asteroida.keys())

Liczba asteroid zidentyfikowanych dzisiaj: 8
Dane ukryte w pierwszej asteroidze:
dict_keys(['links', 'id', 'neo_reference_id', 'name', 'nasa_jpl_url', 'absolute_magnitude_h', 'estimated_diameter', 'is_potentially_hazardous_asteroid', 'close_approach_data', 'is_sentry_object'])


In [13]:
import pandas as pd 
#zmiana listy na df
df = pd.DataFrame(lista_asteroid)

df.head()

,links,id,neo_reference_id,name,nasa_jpl_url,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid,close_approach_data,is_sentry_object
0,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,2454101,2454101,454101 (2013 BP73),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,20.45,{'kilometers': {'estimated_diameter_min': 0.21...,True,"[{'close_approach_date': '2026-05-04', 'close_...",False
1,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3022970,3022970,(1999 SG10),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,20.76,{'kilometers': {'estimated_diameter_min': 0.18...,True,"[{'close_approach_date': '2026-05-04', 'close_...",False
2,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3603631,3603631,(2012 JR4),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,24.00,{'kilometers': {'estimated_diameter_min': 0.04...,False,"[{'close_approach_date': '2026-05-04', 'close_...",False
3,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3763475,3763475,(2016 VL3),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,24.42,{'kilometers': {'estimated_diameter_min': 0.03...,False,"[{'close_approach_date': '2026-05-04', 'close_...",False
4,{'self': 'http://api.nasa.gov/neo/rest/v1/neo/...,3773989,3773989,(2017 HH),https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,19.57,{'kilometers': {'estimated_diameter_min': 0.32...,False,"[{'close_approach_date': '2026-05-04', 'close_...",False


In [16]:
#czyszczenie

#definiujemy listę kolumn, ktore chcemy zostawic
df = pd.DataFrame(lista_asteroid)

potrzebne_kolumny = [
    'name', 
    'absolute_magnitude_h', 
    'is_potentially_hazardous_asteroid', 
    'estimated_diameter', 
    'close_approach_data'
]

#nadpisanie tabeli odfiltrowana wersją

df = df[potrzebne_kolumny]

#wyciagniecie na probe srednicy zeby zobaczyc jak gleboko zagniezdzone sa nasze dane

srednica_pierwszej = df['estimated_diameter'].iloc[0]
print(srednica_pierwszej)

{'kilometers': {'estimated_diameter_min': 0.2160503512, 'estimated_diameter_max': 0.4831032718}, 'meters': {'estimated_diameter_min': 216.0503511964, 'estimated_diameter_max': 483.1032718379}, 'miles': {'estimated_diameter_min': 0.1342474228, 'estimated_diameter_max': 0.3001863631}, 'feet': {'estimated_diameter_min': 708.8266342193, 'estimated_diameter_max': 1584.9845383766}}


In [17]:
#tworzymy nowa kolumne wyciagajac dane z glebi slownika za pomoca lambda, bierzemy najwieksza szacowana wielkosc w metrach
df['diameter_meters_max'] = df['estimated_diameter'].apply(lambda x: x['meters']['estimated_diameter_max'])

#usuniecie starej zagniezdzonej kolumny

df = df.drop('estimated_diameter', axis=1)

df.head()


,name,absolute_magnitude_h,is_potentially_hazardous_asteroid,close_approach_data,diameter_meters_max
0,454101 (2013 BP73),20.45,True,"[{'close_approach_date': '2026-05-04', 'close_...",483.103272
1,(1999 SG10),20.76,True,"[{'close_approach_date': '2026-05-04', 'close_...",418.832119
2,(2012 JR4),24.00,False,"[{'close_approach_date': '2026-05-04', 'close_...",94.197631
3,(2016 VL3),24.42,False,"[{'close_approach_date': '2026-05-04', 'close_...",77.631858
4,(2017 HH),19.57,False,"[{'close_approach_date': '2026-05-04', 'close_...",724.502651


In [18]:
# wyciągamy prędkość w kilometrach na sekundę
# Krok po kroku: wejdź do listy [0] -> wejdź do 'relative_velocity' -> wejdź do 'kilometers_per_second'

df['velocity_km_s'] = df['close_approach_data'].apply(lambda x: x[0]['relative_velocity']['kilometers_per_second'])

#wyciagamy odległość ominięcia Ziemi w kilometrach
df['miss_distance_km'] = df['close_approach_data'].apply(lambda x: x[0]['miss_distance']['kilometers'])

#usuwamy stara zagniezdzona kolumne

df = df.drop('close_approach_data', axis=1)

#konwertujemy nowe kolumny na format liczbowy
# NASA API zwraca te dwie wartości jako tekst (string), co uniemożliwiłoby nam późniejsze wyliczenia i wykresy w Power BI

df['velocity_km_s'] = df['velocity_km_s'].astype(float)
df['miss_distance_km'] = df['miss_distance_km'].astype(float)

df.head()



,name,absolute_magnitude_h,is_potentially_hazardous_asteroid,diameter_meters_max,velocity_km_s,miss_distance_km
0,454101 (2013 BP73),20.45,True,483.103272,21.100664,1.089836e+07
1,(1999 SG10),20.76,True,418.832119,29.468753,4.649900e+07
2,(2012 JR4),24.00,False,94.197631,8.671085,2.317200e+07
3,(2016 VL3),24.42,False,77.631858,6.499519,6.653513e+07
4,(2017 HH),19.57,False,724.502651,26.970799,6.509240e+07


In [19]:
# 1. Zamieniamy dystans na liczbę całkowitą (int), co usunie notacje naukową
df['miss_distance_km'] = df['miss_distance_km'].astype(int)

# 2. Zaokrąglamy prędkość obiektu do 2 miejsc po przecinku dla lepszej czytelności
df['velocity_km_s'] = df['velocity_km_s'].round(2)

# 3. Dodatkowo instruujemy Pandasa, żeby ewentualne inne duże liczby float 
# wyświetlał w standardowym formacie, a nie naukowym
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# 4. Sprawdzamy ostateczny wygląd naszej tabeli
df.head()

,name,absolute_magnitude_h,is_potentially_hazardous_asteroid,diameter_meters_max,velocity_km_s,miss_distance_km
0,454101 (2013 BP73),20.45,True,483.10,21.10,10898357
1,(1999 SG10),20.76,True,418.83,29.47,46498996
2,(2012 JR4),24.00,False,94.20,8.67,23171996
3,(2016 VL3),24.42,False,77.63,6.50,66535128
4,(2017 HH),19.57,False,724.50,26.97,65092395


In [23]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

#wymuszamy ponowne wczytaniie pliku .env
load_dotenv(override=True)

#zaciagamy dane do logowania z .env
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

# Budujemy tzw. Connection String 

connection_string = f"postgresql://{user}:{password}@{host}:{port}/{db_name}"

#tworzymy silnik bazy danych

engine = create_engine(connection_string)

#  Wysyłamy tabelę (df) do bazy pod nazwą 'asteroidy_dzisiaj'
try:
    df.to_sql('asteroidy_dzisiaj', con=engine, if_exists='replace', index=False)
    print("Sukces, dane załadowane w bazie SQL")
except Exception as e:
    print(f"Błąd połączenia lub zapisu: {e}")







Sukces, dane załadowane w bazie SQL
